# Bayesian optimization with BoTorch: a first walkthroughThis notebook builds one **single-objective Bayesian optimization (BO)** iteration. We start with the core workflow; constraints, noisy objectives, batches, and multi-objective BO can come later.## The loop1. Collect evaluated inputs `train_X` and outcomes `train_Y`.2. Fit a Gaussian process (GP) surrogate.3. Use its posterior to estimate outcomes and uncertainty.4. Score possible experiments with an acquisition function.5. Optimize that function inside the allowed search space.6. Run the proposed experiment, add its measured result, and repeat.```textobservations -> GP surrogate -> acquisition -> next experiment      ^                                         |      +------------ add measured outcome --------+```For `n` experiments, `d` input variables, and one outcome, `train_X` has shape `n x d` and `train_Y` has shape `n x 1`.

## Example problemWe will maximize a simple one-variable objective `y = f(x)`, with `x` in `[0, 1]`. This small synthetic example keeps the first model easy to inspect; the same workflow extends to multiple input variables by using one column per variable.For a real experiment, each input column might represent a formulation variable (for example, concentration or solvent fraction), and the outcome might be conductivity, yield, or another measured quantity. Choose the objective direction deliberately: this notebook maximizes.

## 1. Prepare the observationsEach row of `train_X` is one evaluated input, and the matching row of `train_Y` is its observed outcome. The inputs below are already normalized to the unit interval, which is a good default for GP models. For a physical variable with bounds `[a, b]`, map it to `(x - a) / (b - a)` and keep those bounds to convert the recommendation back later.We use double precision for this small GP example and generate deterministic toy observations.

In [ ]:
import torch# GP calculations are generally more stable in double precision.torch.set_default_dtype(torch.double)train_X = torch.linspace(0.0, 1.0, 7).unsqueeze(-1)train_Y = torch.sin(2 * torch.pi * train_X)print("train_X shape:", train_X.shape)  # n x d = 7 x 1print("train_Y shape:", train_Y.shape)  # n x 1

## 2. Build the GP surrogateA Gaussian process places a distribution over plausible objective functions. After seeing the data, its posterior gives a predictive mean and variance at any input.`SingleTaskGP` is a standard choice for one outcome. In this BoTorch version, it uses a constant mean, a scaled Matern kernel, a Gaussian likelihood, and standardizes the outcome by default. The input in this example is already normalized; BoTorch does not automatically rescale the physical input space for us. A warning about the raw outcome scale may still appear before the default outcome transform is applied.

In [ ]:
from botorch.models import SingleTaskGPmodel = SingleTaskGP(train_X, train_Y)print(model)

### A few GP parts to recognize- **Mean function:** the baseline trend before considering nearby observations.- **Kernel:** describes how similarity between inputs relates to similarity between outcomes. The default scaled Matern kernel is a flexible smoothness assumption.- **Lengthscale and outputscale:** learned kernel parameters describing how quickly the function varies and its overall variation.- **Gaussian likelihood:** accounts for observation noise; it has a learned noise level.The model is constructed, but its hyperparameters still need to be fit to the observations.

## 3. Fit the modelThe marginal log likelihood measures how well the GP hyperparameters explain the observed data. Fitting maximizes this quantity (equivalently, minimizes its negative); BoTorch chooses and runs the appropriate optimizer for this model.

In [ ]:
from gpytorch.mlls import ExactMarginalLogLikelihoodfrom botorch.fit import fit_gpytorch_mllmll = ExactMarginalLogLikelihood(model.likelihood, model)fit_gpytorch_mll(mll);

## 4. Inspect the posteriorFor candidate inputs `X`, the GP posterior summarizes predicted outcomes given the data. Its mean is the current best estimate; its variance describes predictive uncertainty.BoTorch model inputs have shape `batch_shape x q x d`: `q` is the number of points considered jointly and `d` is the number of input variables. Here, three points have shape `3 x 1`.

In [ ]:
test_X = torch.tensor([[0.25], [0.50], [0.75]])

In [ ]:
posterior = model.posterior(test_X)print("Posterior mean:\n", posterior.mean)print("Posterior variance:\n", posterior.variance)

## 5. Score candidates with an acquisition functionAn acquisition function uses the model's predictions to score where it may be valuable to run the next experiment. It balances predicted performance with uncertainty, according to the chosen rule; it does not run the experiment.For a maximization problem, **Expected Improvement (EI)** favors candidates expected to exceed the best observed outcome. **LogEI** is a numerically more stable version, often preferred when optimizing EI. LogEI scores may be negative because they are on a log scale; a larger score is still better. Acquisition functions expect candidates shaped `batch_shape x q x d`. To score each row of `test_X` as a separate one-point batch, insert the `q` dimension with `unsqueeze(-2)`.

In [ ]:
from botorch.acquisition import PosteriorMean# Posterior mean alone is a simple baseline; it does not reward uncertainty.posterior_mean_acq = PosteriorMean(model=model, maximize=True)mean_scores = posterior_mean_acq(test_X.unsqueeze(-2))print("Posterior-mean scores:", mean_scores)

In [ ]:
from botorch.acquisition import ExpectedImprovementbest_f = train_Y.max()ei = ExpectedImprovement(model=model, best_f=best_f)ei_scores = ei(test_X.unsqueeze(-2))print("Best observed outcome:", best_f)print("EI scores:", ei_scores)

For maximization, EI is```mathEI(x) = E[max(f(x) - f_best, 0)]```where `f_best = max(train_Y)`. A high score means the model expects more improvement, accounting for both the predictive mean and uncertainty. Posterior mean alone can miss uncertain but promising regions; EI addresses that exploration/exploitation trade-off.

In [ ]:
from botorch.acquisition import LogExpectedImprovementacq = LogExpectedImprovement(model=model, best_f=best_f)log_ei_scores = acq(test_X.unsqueeze(-2))print("LogEI scores:", log_ei_scores)

## 6. Optimize the acquisition functionRather than score only a hand-picked grid, `optimize_acqf` searches the continuous bounded domain for a high-scoring candidate. Its `bounds` must use the same normalized coordinates as the model's training inputs.Here `q=1` asks for one next experiment. Acquisition optimization is non-convex, so BoTorch uses sampled initial points and multiple local restarts.

In [ ]:
from botorch.optim import optimize_acqfbounds = torch.tensor([[0.0], [1.0]])  # lower and upper bounds, one row per input dimensioncandidate, acq_value = optimize_acqf(    acq_function=acq,    bounds=bounds,    q=1,    num_restarts=10,    raw_samples=100,)print("Proposed normalized input:", candidate)print("LogEI at proposal:", acq_value)

### What the optimizer is doing1. Sample possible starting points within `bounds`.2. Select promising starts.3. Locally optimize the acquisition function from multiple starts.4. Return the candidate with the best acquisition value found.The returned candidate is a **proposal**, not a measured result. In a real workflow, map it back to physical units, run the experiment, record its outcome, append the new `(X, Y)` pair, then refit and optimize again.

## 7. The next BO iterationAfter evaluating the proposed experiment at `candidate`, suppose the measured outcome is `new_Y`. The basic loop is:```pythontrain_X = torch.cat([train_X, candidate])train_Y = torch.cat([train_Y, new_Y])```Then refit the GP using the updated data and find the next proposal. In practice, also track units, experimental failures, replicate measurements, and whether the objective is to maximize or minimize.### Key takeaways- The GP models the unknown objective and its uncertainty.- The acquisition function turns that posterior into a search policy.- Acquisition optimization proposes where to evaluate next.- Only a real measurement (or a simulator result) provides the next observed outcome.